# Titanic dataset - numpy, pandas, matplotlib, scikit-learn

## imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## pandas

In [ ]:
df = pd.read_csv("Titanic-Dataset.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

Age has 177 nulls, Cabin has way too many and Embarked has just 2.

In [ ]:
# fill age with median, embarked with most common value, drop cabin
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df = df.drop("Cabin", axis=1)

df.isnull().sum().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df[["Name", "Sex", "Age", "Survived"]].head()

In [ ]:
# first class women who survived
df[(df["Sex"] == "female") & (df["Pclass"] == 1) & (df["Survived"] == 1)].head()

In [ ]:
df.sort_values("Fare", ascending=False)[["Name", "Pclass", "Fare"]].head()

In [ ]:
df.groupby("Sex")["Survived"].mean()

In [ ]:
df.groupby("Pclass")["Survived"].mean()

In [ ]:
df.groupby("Pclass").agg({"Age": "mean", "Fare": "mean", "Survived": "mean"})

In [ ]:
pd.pivot_table(df, values="Survived", index="Sex", columns="Pclass", aggfunc="mean")

In [ ]:
pd.crosstab(df["Embarked"], df["Survived"])

In [ ]:
# new columns
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["Title"] = df["Name"].str.extract(r", ([^\.]+)\.", expand=False)

df["Title"].value_counts().head(6)

In [ ]:
# keep the 4 common titles, rest goes to Other
df["Title"] = df["Title"].where(df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Other")
df.groupby("Title")["Survived"].mean()

In [ ]:
df["AgeGroup"] = pd.cut(df["Age"], bins=[0, 12, 18, 35, 60, 100],
                        labels=["child", "teen", "young adult", "adult", "senior"])
df["AgeGroup"].value_counts()

## numpy

In [ ]:
age = df["Age"].to_numpy()
fare = df["Fare"].to_numpy()
print(type(age), age.shape)

In [ ]:
print("mean", age.mean())
print("median", np.median(age))
print("std", age.std())
print("min", age.min(), "max", age.max())
print(np.percentile(age, [25, 50, 75]))

In [ ]:
# how many kids
kids = age[age < 18]
print(len(kids), len(kids) / len(age) * 100)

In [ ]:
df["IsChild"] = np.where(df["Age"] < 18, 1, 0)
df["IsChild"].value_counts()

In [ ]:
# fare is very skewed, log helps
fare_log = np.log1p(fare)
print(fare.mean(), np.median(fare))
print(fare_log.mean(), np.median(fare_log))

In [ ]:
age_scaled = (age - age.mean()) / age.std()
print(age_scaled.mean().round(4), age_scaled.std())

In [ ]:
arr = df[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]].to_numpy()
np.corrcoef(arr.T).round(2)

In [ ]:
both = np.column_stack([age, fare])
print(both.shape)
print(both.mean(axis=0))
print(both.max(axis=0))

In [ ]:
counts, edges = np.histogram(age, bins=np.arange(0, 90, 10))
print(counts)
print(edges)

## matplotlib

In [ ]:
counts = df["Survived"].value_counts().sort_index()
plt.bar(["Died", "Survived"], counts.values, color=["red", "green"])
plt.title("Survived vs died")
plt.ylabel("count")
plt.show()

In [ ]:
sex = df["Sex"].value_counts()
plt.pie(sex, labels=sex.index, autopct="%1.1f%%")
plt.title("Male vs female")
plt.show()

In [ ]:
plt.hist(df["Age"], bins=20, edgecolor="black")
plt.title("Age distribution")
plt.xlabel("age")
plt.ylabel("passengers")
plt.show()

In [ ]:
df.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="mean").plot(kind="bar")
plt.title("Survival rate by class and sex")
plt.ylabel("survival rate")
plt.xticks(rotation=0)
plt.show()

In [ ]:
fares = [df[df["Pclass"] == c]["Fare"] for c in [1, 2, 3]]
plt.boxplot(fares, labels=["1st", "2nd", "3rd"])
plt.title("Fare by class")
plt.show()

In [ ]:
plt.scatter(df["Age"], df["Fare"], c=df["Survived"], cmap="coolwarm", alpha=0.5)
plt.xlabel("age")
plt.ylabel("fare")
plt.title("Age vs fare")
plt.show()

In [ ]:
cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]
corr = df[cols].corr()

plt.figure(figsize=(6, 5))
plt.imshow(corr, cmap="coolwarm")
plt.colorbar()
plt.xticks(range(len(cols)), cols, rotation=45)
plt.yticks(range(len(cols)), cols)
plt.title("Correlation")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

df.groupby("Embarked")["Survived"].mean().plot(kind="bar", ax=ax[0])
ax[0].set_title("by port")

df.groupby("FamilySize")["Survived"].mean().plot(marker="o", ax=ax[1])
ax[1].set_title("by family size")

df.groupby("Title")["Survived"].mean().plot(kind="bar", ax=ax[2])
ax[2].set_title("by title")

plt.tight_layout()
plt.show()

## scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# pick columns and turn text columns into numbers
data = df[["Survived", "Pclass", "Sex", "Age", "Fare", "Embarked", "FamilySize"]]
data = pd.get_dummies(data, columns=["Sex", "Embarked"], drop_first=True)
data.head()

In [ ]:
X = data.drop("Survived", axis=1)
y = data["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

In [ ]:
# scale age and fare (fit only on train)
scaler = StandardScaler()
cols_to_scale = ["Age", "Fare"]

X_train_s = X_train.copy()
X_test_s = X_test.copy()
X_train_s[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_s[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
print("logistic regression:", accuracy_score(y_test, lr.predict(X_test_s)))

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
print("decision tree:", accuracy_score(y_test, dt.predict(X_test)))

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print("random forest:", accuracy_score(y_test, rf.predict(X_test)))

In [ ]:
# cross validation on the logistic regression
scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)
print(scores)
print(scores.mean())

In [ ]:
pred = rf.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot(kind="barh")
plt.title("Random forest feature importance")
plt.show()